[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/dif/u5_reglas.ipynb)

# Dif. U5 · Reglas de derivación
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 5 del bloque de cálculo diferencial. Tiene una sección por subtema, en el mismo orden que el libro:

1. Reglas de constante, potencia y suma
2. Reglas de producto y cociente
3. Regla de la cadena
4. Derivadas de orden superior
5. Derivadas de exponencial, logaritmo y funciones trigonométricas
6. Derivación implícita
7. Derivación logarítmica

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección termina con una celda **Contrasta**: antes de ejecutarla, deriva a mano el caso que se indica y escribe tu resultado. Comprueba siempre una derivada en al menos dos puntos y evita los que simplifican demasiado ($x=0$, $x=1$): en ellos una fórmula equivocada puede coincidir con la correcta.

**Etiqueta del repositorio.** `DIF-U5` (bloque, unidad). Las celdas de código de cada sección vienen de `dif/u5_reglas/NN_*.py`; los fragmentos que el libro imprime, de `dif/u5_reglas/fragmentos/`.

**Convenciones.** Punto decimal. Los valores se *redondean* (no se truncan) a las cifras significativas que elijas. Para escribir expresiones usa `x` como variable, `*` para multiplicar, `**` o `^` para potencias, `sqrt(...)` para raíces cuadradas, `exp(...)` para $e^x$ y `log(...)` para el logaritmo natural. Los ángulos de `sin`, `cos` y `tan` van en **radianes**, salvo en la calculadora de la sección 5, que tiene un selector.

## 0. Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8"

import sys, math, re
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
import ipywidgets as widgets
from IPython.display import display
from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                        implicit_multiplication_application, convert_xor, rationalize)

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Símbolos y utilidades compartidas por todo el notebook
x, y, u = sp.symbols("x y u", real=True)

_FUNCIONES = {"sqrt": sp.sqrt, "abs": sp.Abs, "exp": sp.exp, "log": sp.log,
              "pi": sp.pi, "sin": sp.sin, "cos": sp.cos, "tan": sp.tan, "E": sp.E}
_GLOBAL = {"Integer": sp.Integer, "Float": sp.Float, "Rational": sp.Rational,
           "Symbol": sp.Symbol, "Function": sp.Function}
# rationalize: 0.1 se lee como 1/10, para que las derivadas salgan exactas
_TRANSF = standard_transformations + (implicit_multiplication_application, convert_xor, rationalize)
_SIMBOLOS = {"x": x, "y": y, "u": u}


def parsear(texto, variables=("x",), funciones=None):
    """Convierte un texto en expresión de sympy o explica qué salió mal.
    variables: nombres permitidos (x, y o u). funciones: reemplaza el diccionario de funciones."""
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("Escribe una expresión, por ejemplo: x**3 - 3*x")
    fun = dict(_FUNCIONES if funciones is None else funciones)
    local = {**fun, **{v: _SIMBOLOS[v] for v in variables}}
    desconocidos = sorted(set(re.findall(r"[A-Za-z_]\w*", texto)) - set(local))
    if desconocidos:
        raise ValueError(f"No reconozco {desconocidos}. Usa {', '.join(variables)} como variable y, si hace falta, "
                         "sqrt, exp, log (natural), sin, cos, tan o pi.")
    try:
        expr = parse_expr(texto, local_dict=local, global_dict=dict(_GLOBAL), transformations=_TRANSF)
    except Exception as err:
        raise ValueError("No pude leer la expresión. Usa * para multiplicar, ** o ^ para potencias "
                         "y sqrt(...) para raíces.") from err
    if not isinstance(expr, sp.Expr):
        raise ValueError("Escribe una expresión, no una ecuación (sin el signo =).")
    if expr.atoms(sp.core.function.AppliedUndef):
        raise ValueError("Hay una función que no conozco. Las permitidas son sqrt, exp, log, sin, cos y tan.")
    libres = {str(s) for s in expr.free_symbols} - set(variables)
    if libres:
        raise ValueError(f"Símbolos no permitidos: {sorted(libres)}. Solo puedes usar {list(variables)}.")
    return expr


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas y devuelve texto."""
    v = float(valor)
    if math.isnan(v):
        return "no definido"
    if math.isinf(v):
        return "+∞" if v > 0 else "-∞"
    if v == 0:
        return "0"
    return np.format_float_positional(v, precision=n, unique=False, fractional=False, trim="-")


def iguales(e1, e2):
    """True si las dos expresiones son equivalentes."""
    return sp.simplify(sp.sympify(e1) - sp.sympify(e2)) == 0


def evaluar_en(expr, var, a):
    """Valor exacto de expr en var = a; ValueError si no es un número real finito."""
    v = sp.simplify(expr.subs(var, sp.nsimplify(a)))
    if not (v.is_real and v.is_finite):
        raise ValueError(f"la expresión no tiene valor real en {var} = {a} (fuera del dominio).")
    return v


def cociente_num(expr, var, a, hh=1e-6):
    """Cociente incremental numérico (f(a + h) - f(a))/h, como comprobación independiente."""
    f = sp.lambdify(var, expr, "math")
    return (f(float(a) + hh) - f(float(a))) / hh


def cerca(p, q, tol=1e-9):
    return abs(float(p) - float(q)) < tol


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una calculadora rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False


def contrasta(mi_texto, referencia, variables=("x",), pista=""):
    """Celda Contrasta: compara tu resultado a mano (texto) con el de la calculadora."""
    if mi_texto is None or str(mi_texto).strip() == "":
        print("Falta tu cálculo a mano. Escríbelo arriba y vuelve a ejecutar la celda.")
        return None
    try:
        mio = parsear(str(mi_texto), variables)
    except ValueError as err:
        print(err); return None
    ok = iguales(mio, referencia)
    print("Coinciden." if ok else "No coinciden. " + pista)
    return ok

## 1. Reglas de constante, potencia y suma

| Regla | Fórmula | Condición |
|---|---|---|
| constante | $(c)'=0$ | |
| potencia | $(x^n)'=n\,x^{n-1}$ | $n$ entero ($x\neq0$ si $n<0$) o $n$ real con $x>0$; exponente **constante** |
| múltiplo constante | $(c\,f)'=c\,f'$ | |
| suma y resta | $(f\pm g)'=f'\pm g'$ | |

Antes de derivar, pasa raíces y cocientes a potencias: $\sqrt{x}=x^{1/2}$, $5/x^4=5x^{-4}$.

La calculadora deriva una suma de potencias término por término y dice qué regla usó en cada uno. Si un término no es de la forma $c\,x^n$ con $n$ constante (por ejemplo $2^x$, $x^x$ o $x\,e^x$), lo rechaza: necesita otra regla.

In [ ]:
def terminos_potencia(f):
    """Lista de (término, regla, derivada) de una suma de términos c*x**n con n constante."""
    filas = []
    for t in sp.Add.make_args(sp.expand(f)):
        if not t.has(x):
            filas.append((t, "constante", sp.Integer(0)))
            continue
        c, n = t.as_coeff_exponent(x)
        if c.has(x) or n.has(x):
            raise ValueError(f"el término {t} no es de la forma c·x^n con exponente constante: "
                             "necesita la regla del producto, de la cadena o la derivación logarítmica.")
        regla = "potencia" if c == 1 else "múltiplo constante y potencia"
        filas.append((t, regla, sp.simplify(c * n * x**(n - 1))))
    return filas


def calculadora_potencia(f_txt, a, n_cifras):
    try:
        f = parsear(f_txt)
        filas = terminos_potencia(f)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    for t, regla, d in filas:
        print(f"  {str(t):>16}  ->  {str(d):<18} ({regla})")
    total = sp.Add(*[d for _, _, d in filas])
    print("f'(x) =", total, "   (regla de la suma)")
    print("comprobación con sp.diff:", "coincide" if iguales(total, sp.diff(f, x)) else "NO coincide")
    try:
        va = evaluar_en(total, x, a)
        print(f"f'({a}) = {va} ≈ {cifras(va, n_cifras)}   | cociente con h = 1e-6: {cifras(cociente_num(f, x, a), n_cifras)}"
              f"   ({n_cifras} cifras significativas)")
    except (ValueError, ZeroDivisionError) as err:
        print("En a:", err)


widgets.interact(calculadora_potencia,
    f_txt=widgets.Text(value="4*x**5 - 3/x**2 + 7*sqrt(x) - 2", description="f(x) ="),
    a=widgets.FloatText(value=1.0, description="a"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 1 (resultado conocido)
def _derivada_terminos(texto):
    return sp.Add(*[d for _, _, d in terminos_potencia(parsear(texto))])

PRUEBAS_1 = [
    ("4x^5 - 3/x^2 + 7√x - 2 -> 20x^4 + 6/x^3 + 7/(2√x)",
     lambda: iguales(_derivada_terminos("4*x**5 - 3/x**2 + 7*sqrt(x) - 2"), 20*x**4 + 6/x**3 + sp.Rational(7, 2)/sp.sqrt(x))),
    ("x^3 - 3x -> 3x^2 - 3", lambda: iguales(_derivada_terminos("x**3 - 3*x"), 3*x**2 - 3)),
    ("constante 7 -> 0", lambda: _derivada_terminos("7") == 0),
    ("6√x - 2/x en 1: 5", lambda: _derivada_terminos("6*sqrt(x) - 2/x").subs(x, 1) == 5),
    ("2^x se rechaza (exponente variable)", lambda: _rechaza(lambda: terminos_potencia(parsear("2**x")))),
    ("x·e^x se rechaza (producto)", lambda: _rechaza(lambda: terminos_potencia(parsear("x*exp(x)")))),
]
for nombre, prueba in PRUEBAS_1:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 1).** Deriva a mano $f(x)=2x^4-\dfrac{5}{x}+3\sqrt{x}$. Escribe tu resultado como texto (con el formato de `"3*x**2 + 1/x"`) y compáralo con la calculadora.

In [ ]:
mi_derivada = None       # escribe un texto entre comillas

contrasta(mi_derivada, sp.diff(parsear("2*x**4 - 5/x + 3*sqrt(x)"), x),
          pista="¿Pasaste 5/x a 5x^(-1) antes de derivar? ¿El signo del segundo término quedó positivo?")

## 2. Reglas de producto y cociente

$$\big(u\,v\big)'=u'\,v+u\,v',\qquad \left(\frac{u}{v}\right)'=\frac{u'\,v-u\,v'}{v^2}\quad (v\neq0).$$

La derivada de un producto **no** es el producto de las derivadas: en el rectángulo de lados $u$ y $v$, el área cambia por dos tiras, $v\,\Delta u$ y $u\,\Delta v$, y una esquina $\Delta u\,\Delta v$ que desaparece en el límite. En el cociente, el orden del numerador importa: invertirlo cambia el signo.

La calculadora recibe $u(x)$ y $v(x)$, arma $(uv)'$ y $(u/v)'$ con las reglas, compara con `sp.diff` y avisa dónde $v(x)=0$.

In [ ]:
def producto_cociente(u_txt, v_txt):
    """Diccionario con u', v', (uv)' y (u/v)' por las reglas, y los ceros reales de v."""
    U, V = parsear(u_txt), parsear(v_txt)
    if V == 0:
        raise ValueError("v(x) = 0 para todo x: el cociente no está definido.")
    dU, dV = sp.diff(U, x), sp.diff(V, x)
    prod = sp.simplify(dU * V + U * dV)
    coc = sp.simplify((dU * V - U * dV) / V**2)
    ceros = sorted(sp.solveset(V, x, domain=sp.S.Reals), key=float) if V.is_polynomial(x) else None
    return {"u'": dU, "v'": dV, "(uv)'": prod, "(u/v)'": coc, "u'v'": sp.simplify(dU * dV), "ceros_v": ceros,
            "ok": iguales(prod, sp.diff(U * V, x)) and iguales(coc, sp.diff(U / V, x))}


def calculadora_producto(u_txt, v_txt):
    try:
        r = producto_cociente(u_txt, v_txt)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print("u' =", r["u'"], "     v' =", r["v'"])
    print("(uv)'  = u'v + uv'          =", r["(uv)'"])
    print("(u/v)' = (u'v - uv')/v^2    =", r["(u/v)'"])
    print("u'v' (NO es la derivada de uv) =", r["u'v'"])
    if r["ceros_v"] is None:
        print("v no es polinomio: revisa a mano dónde se anula.")
    elif r["ceros_v"]:
        print("el cociente no está definido donde v = 0: x =", ", ".join(str(c) for c in r["ceros_v"]))
    print("comprobación con sp.diff:", "coinciden" if r["ok"] else "NO coinciden")


widgets.interact(calculadora_producto,
    u_txt=widgets.Text(value="x**2 + 1", description="u(x) ="),
    v_txt=widgets.Text(value="x**3 - x", description="v(x) ="));

In [ ]:
# Casos de prueba de la sección 2 (resultado conocido)
PRUEBAS_2 = [
    ("(x²+1)(x³-x) -> 5x^4 - 1", lambda: iguales(producto_cociente("x**2 + 1", "x**3 - x")["(uv)'"], 5*x**4 - 1)),
    ("en x = 2: u'v + uv' = 79, u'v' = 44",
     lambda: producto_cociente("x**2 + 1", "x**3 - x")["(uv)'"].subs(x, 2) == 79
             and producto_cociente("x**2 + 1", "x**3 - x")["u'v'"].subs(x, 2) == 44),
    ("(x+1)/(x-1) -> -2/(x-1)²", lambda: iguales(producto_cociente("x + 1", "x - 1")["(u/v)'"], -2/(x - 1)**2)),
    ("ceros de v = x - 1: [1]", lambda: producto_cociente("x + 1", "x - 1")["ceros_v"] == [1]),
    ("costo promedio C/q en 100: -0.15",
     lambda: producto_cociente("2000 + 15*x + 0.05*x**2", "x")["(u/v)'"].subs(x, 100) == sp.Rational(-3, 20)),
    ("v = x² + 1 no tiene ceros reales", lambda: producto_cociente("1", "x**2 + 1")["ceros_v"] == []),
    ("v = 0 se rechaza con mensaje", lambda: _rechaza(lambda: producto_cociente("x", "0"))),
]
for nombre, prueba in PRUEBAS_2:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 2).** Deriva a mano $\dfrac{x^2}{x+1}$ con la regla del cociente y escribe el resultado como texto (con el formato de `"(x - 1)/(x + 3)**2"`).

In [ ]:
mi_derivada = None       # escribe un texto entre comillas

contrasta(mi_derivada, sp.diff(x**2 / (x + 1), x),
          pista="¿Escribiste u'v - uv' en ese orden? ¿Elevaste v al cuadrado en el denominador?")

## 3. Regla de la cadena

Si $y=f(u)$ y $u=g(x)$,
$$\frac{dy}{dx}=\frac{dy}{du}\,\frac{du}{dx}=f'\big(g(x)\big)\,g'(x).$$
Se deriva la función de afuera evaluada en la de adentro y se multiplica por la derivada de la de adentro.

**Retoma tu predicción de la semilla.** La apuesta $10(x^2+1)^9$ da $5120$ en $x=1$. La celda siguiente calcula el cociente incremental con $h=0.001$ y la derivada con la regla de la cadena. Compara con lo que anotaste.

In [ ]:
f_sem = (x**2 + 1)**10
print("cociente con h = 0.001:", round(cociente_num(f_sem, x, 1, 1e-3), 1))
print("apuesta 10(x²+1)^9    :", (10*(x**2 + 1)**9).subs(x, 1))
print("regla de la cadena    :", sp.diff(f_sem, x), "->", sp.diff(f_sem, x).subs(x, 1))

La calculadora recibe la función de afuera $f(u)$, escrita con la variable `u`, y la de adentro $g(x)$. Muestra cada paso, compara con `sp.diff` de la composición y evalúa en un punto junto al cociente incremental.

In [ ]:
def cadena(f_txt, g_txt, a=None):
    """Pasos de la regla de la cadena para f(g(x)). Si se da a, también el valor en a."""
    F = parsear(f_txt, variables=("u",))
    G = parsear(g_txt, variables=("x",))
    dF, dG = sp.diff(F, u), sp.diff(G, x)
    resultado = sp.simplify(dF.subs(u, G) * dG)
    pasos = {"f'(u)": dF, "g'(x)": dG, "f'(g(x))": dF.subs(u, G), "(f(g(x)))'": resultado,
             "ok": iguales(resultado, sp.diff(F.subs(u, G), x))}
    if a is not None:
        ga = evaluar_en(G, x, a)
        try:
            evaluar_en(F, u, ga)
        except ValueError:
            raise ValueError(f"g({a}) = {ga} queda fuera del dominio de f.") from None
        pasos["valor"] = evaluar_en(resultado, x, a)
        pasos["cociente"] = cociente_num(F.subs(u, G), x, a)
    return pasos


def calculadora_cadena(f_txt, g_txt, a, n_cifras):
    try:
        p = cadena(f_txt, g_txt, a)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print("f'(u)       =", p["f'(u)"])
    print("g'(x)       =", p["g'(x)"])
    print("f'(g(x))    =", p["f'(g(x))"])
    print("derivada    =", p["(f(g(x)))'"], "  ->", "coincide con sp.diff" if p["ok"] else "NO coincide")
    print(f"en x = {a}: {cifras(p['valor'], n_cifras)}   | cociente con h = 1e-6: {cifras(p['cociente'], n_cifras)}"
          f"   ({n_cifras} cifras significativas; ángulos en radianes)")


widgets.interact(calculadora_cadena,
    f_txt=widgets.Text(value="sqrt(u)", description="f(u) ="),
    g_txt=widgets.Text(value="144*x**2 + 2500", description="g(x) ="),
    a=widgets.FloatText(value=10.0, description="a"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 3 (resultado conocido; los cuatro primeros son los del prompt del libro)
PRUEBAS_3 = [
    ("f = u^5, g = 3x + 1, a = 0: 15", lambda: cadena("u**5", "3*x + 1", 0)["valor"] == 15),
    ("f = √u, g = x² + 9, a = 4: 0.8", lambda: cadena("sqrt(u)", "x**2 + 9", 4)["valor"] == sp.Rational(4, 5)),
    ("f = u^10, g = x² + 1, a = 1: 10240", lambda: cadena("u**10", "x**2 + 1", 1)["valor"] == 10240),
    ("f = 1/u³, g = 2x - 1, a = 1: -6", lambda: cadena("1/u**3", "2*x - 1", 1)["valor"] == -6),
    ("dron: √u con 144x² + 2500 en 10: 144/13", lambda: cadena("sqrt(u)", "144*x**2 + 2500", 10)["valor"] == sp.Rational(144, 13)),
    ("g(a) fuera del dominio de f se rechaza", lambda: _rechaza(lambda: cadena("sqrt(u)", "-x**2 - 1", 0))),
    ("f escrita con x se rechaza", lambda: _rechaza(lambda: cadena("x**2", "3*x", 1))),
]
for nombre, prueba in PRUEBAS_3:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 3).** Deriva a mano $\sqrt{1+4x^2}$ y evalúa en $x=1$. Escribe el número con 4 cifras significativas.

In [ ]:
mi_valor = None          # escribe un número

ref = cadena("sqrt(u)", "1 + 4*x**2", 1)["valor"]
if mi_valor is None:                     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    print("La calculadora da:", ref, "≈", cifras(ref, 4))
    print("coincide" if abs(mi_valor - float(ref)) < 5e-4 else "NO coincide: ¿multiplicaste por la derivada de 1 + 4x²?")

## 4. Derivadas de orden superior

$f''=(f')'$, $f'''=(f'')'$ y, en general, $f^{(n)}$. Con $y=f(x)$: $\dfrac{d^2y}{dx^2},\ \dfrac{d^3y}{dx^3},\dots$ Las unidades son las de $y$ entre las de $x$ elevadas a la $n$.

| Posición $s(t)$ | $s'$ | $s''$ | $s'''$ |
|---|---|---|---|
| m | velocidad, m/s | aceleración, m/s² | sacudida (*jerk*), m/s³ |

En una viga (modelo de Euler-Bernoulli): deflexión $y$, giro $y'$, momento $M=EI\,y''$, cortante $V=EI\,y'''$ y carga $EI\,y''''=-w$.

La calculadora tabula $f, f', \dots, f^{(n)}$ y sus valores en un punto, y avisa cuando la derivada se vuelve cero.

In [ ]:
def derivadas_sucesivas(f, n):
    """[f, f', ..., f^(n)] como expresiones simplificadas."""
    if not (isinstance(n, int) and n >= 1):
        raise ValueError("el orden n debe ser un entero mayor o igual que 1.")
    lista = [f]
    for _ in range(n):
        lista.append(sp.simplify(sp.diff(lista[-1], x)))
    return lista


def calculadora_orden(f_txt, n, a, n_cifras):
    try:
        lista = derivadas_sucesivas(parsear(f_txt), int(n))
    except ValueError as err:
        print("Revisa la entrada:", err); return
    for k, d in enumerate(lista):
        nombre = "f" + ("'" * k if k <= 3 else f"^({k})")
        try:
            val = cifras(evaluar_en(d, x, a), n_cifras)
        except ValueError:
            val = "no definida"
        print(f"{nombre:>7}(x) = {str(d):<32} en x = {a}: {val}")
        if d == 0:
            print("        desde aquí todas las derivadas son cero."); break


widgets.interact(calculadora_orden,
    f_txt=widgets.Text(value="2*x**5 - x**3 + 4*x", description="f(x) ="),
    n=widgets.IntSlider(value=6, min=1, max=8, description="orden n"),
    a=widgets.FloatText(value=1.0, description="a"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 4 (resultado conocido)
PRUEBAS_4 = [
    ("2x^5 - x^3 + 4x: quinta 240 y sexta 0",
     lambda: derivadas_sucesivas(parsear("2*x**5 - x**3 + 4*x"), 6)[5:] == [240, 0]),
    ("1/x: tercera -6/x^4", lambda: iguales(derivadas_sucesivas(1/x, 3)[3], -6/x**4)),
    ("4.9t²: segunda 9.8 (aceleración)", lambda: derivadas_sucesivas(parsear("4.9*x**2"), 2)[2] == sp.Rational(49, 5)),
    ("ménsula w = 2, L = 3, EI = 2000: EI·y'''' = -2",
     lambda: 2000 * derivadas_sucesivas(parsear("-(2/48000)*(x**4 - 12*x**3 + 54*x**2)"), 4)[4] == -2),
    ("x^4 - 6x²: segunda en 1 vale 0", lambda: derivadas_sucesivas(x**4 - 6*x**2, 2)[2].subs(x, 1) == 0),
    ("orden 0 se rechaza", lambda: _rechaza(lambda: derivadas_sucesivas(x**2, 0))),
]
for nombre, prueba in PRUEBAS_4:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 4).** Calcula a mano $f''(1)$ para $f(x)=x^3-2x^2+5$ y escribe el número.

In [ ]:
mi_valor = None          # escribe un número

ref = derivadas_sucesivas(x**3 - 2*x**2 + 5, 2)[2].subs(x, 1)
if mi_valor is None:                     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    print("La calculadora da:", ref)
    print("coincide" if mi_valor == ref else "NO coincide: deriva dos veces antes de sustituir x = 1")

## 5. Derivadas de exponencial, logaritmo y funciones trigonométricas

Con $x$ en **radianes** en las trigonométricas:

| $f$ | $e^x$ | $a^x$ | $\ln x$ | $\mathrm{sen}\,x$ | $\cos x$ | $\tan x$ |
|---|---|---|---|---|---|---|
| $f'$ | $e^x$ | $a^x\ln a$ | $1/x$ ($x>0$) | $\cos x$ | $-\mathrm{sen}\,x$ | $1/\cos^2x$ |

Con grados, $\mathrm{sen}(\theta^\circ)=\mathrm{sen}(\pi\theta/180)$ y la cadena agrega el factor $\pi/180\approx0.01745$.

La calculadora tiene un selector visible de unidad angular. En grados reescribe cada `sin`, `cos` y `tan` con el argumento multiplicado por $\pi/180$ y muestra el resultado. Rechaza `log` de valores no positivos.

In [ ]:
_GRADOS = {**_FUNCIONES,
           "sin": lambda e: sp.sin(sp.pi * e / 180), "cos": lambda e: sp.cos(sp.pi * e / 180),
           "tan": lambda e: sp.tan(sp.pi * e / 180)}


def derivar_angulos(texto, modo="radianes"):
    """(f, f') con los ángulos en el modo indicado ('radianes' o 'grados')."""
    if modo not in ("radianes", "grados"):
        raise ValueError("el modo debe ser 'radianes' o 'grados'.")
    f = parsear(texto, funciones=_GRADOS if modo == "grados" else None)
    return f, sp.diff(f, x)


def derivada_en(texto, a, modo="radianes"):
    """f'(a); ValueError si a queda fuera del dominio (por ejemplo, log de un valor no positivo)."""
    f, d = derivar_angulos(texto, modo)
    for arg in [g.args[0] for g in f.atoms(sp.log)]:
        if not evaluar_en(arg, x, a) > 0:
            raise ValueError(f"en x = {a} el argumento del logaritmo, {arg}, no es positivo.")
    evaluar_en(f, x, a)
    return evaluar_en(d, x, a)


def calculadora_exp_trig(f_txt, modo, a, n_cifras):
    try:
        f, d = derivar_angulos(f_txt, modo)
        va = derivada_en(f_txt, a, modo)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    unidad = ("por grado" if modo == "grados" else "por radián") if f.has(sp.sin, sp.cos, sp.tan) else ""
    print(f"modo angular: {modo.upper()}")
    print("f(x)  =", f)
    print("f'(x) =", sp.simplify(d))
    print(f"f'({a}) = {cifras(va, n_cifras)} {unidad}  ({n_cifras} cifras significativas)"
          f"   | cociente con h = 1e-6: {cifras(cociente_num(f, x, a), n_cifras)}")


widgets.interact(calculadora_exp_trig,
    f_txt=widgets.Text(value="sin(x)", description="f(x) ="),
    modo=widgets.ToggleButtons(options=["radianes", "grados"], description="ángulos"),
    a=widgets.FloatText(value=60.0, description="a"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 5 (resultado conocido)
PRUEBAS_5 = [
    ("e^x -> e^x", lambda: derivar_angulos("exp(x)")[1] == sp.exp(x)),
    ("2^x en 0: ln 2", lambda: iguales(derivada_en("2**x", 0), sp.log(2))),
    ("sen x en π/3 (radianes): 1/2", lambda: derivada_en("sin(x)", sp.pi/3) == sp.Rational(1, 2)),
    ("sen x en 60 (grados): π/360 ≈ 0.008727", lambda: iguales(derivada_en("sin(x)", 60, "grados"), sp.pi/360)),
    ("café 20 + 70e^(-0.1x) en 5: -4.2457", lambda: cerca(derivada_en("20 + 70*exp(-0.1*x)", 5), -4.2457146, 1e-6)),
    ("sigmoide en 0: 0.25", lambda: derivada_en("1/(1 + exp(-x))", 0) == sp.Rational(1, 4)),
    ("ln x en -1 se rechaza", lambda: _rechaza(lambda: derivada_en("log(x)", -1))),
    ("e^(-x) cos 2x en 0: -1", lambda: derivada_en("exp(-x)*cos(2*x)", 0) == -1),
]
for nombre, prueba in PRUEBAS_5:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 5).** Deriva a mano $e^{-2x}\,\mathrm{sen}\,3x$, evalúa en $x=0$ y escribe el número. Después cambia la calculadora a grados con la misma función y $a=0$: ¿por qué cambia el resultado?

In [ ]:
mi_valor = None          # escribe un número

ref = derivada_en("exp(-2*x)*sin(3*x)", 0)
if mi_valor is None:                     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    print("La calculadora (radianes) da:", ref, "| en grados:", cifras(derivada_en("exp(-2*x)*sin(3*x)", 0, "grados"), 4))
    print("coincide" if mi_valor == ref else "NO coincide: regla del producto y cadena con la interior 3x")

## 6. Derivación implícita

Para una curva dada por una ecuación en $x$ y $y$: se derivan los dos lados respecto a $x$ tratando $y$ como $y(x)$ (cada término con $y$ lleva un factor $y'$ por la cadena; $xy$ se deriva con la regla del producto) y se despeja $y'$. La pendiente depende del **punto**, no solo de $x$. Donde el coeficiente de $y'$ se anula, la fórmula no da la pendiente y puede haber una tangente vertical.

La calculadora recibe la ecuación con un signo `=` y un punto. Comprueba que el punto esté en la curva y da $y'$, su valor y la recta tangente.

In [ ]:
def implicita(ecuacion, px, py):
    """(y' como expresión, valor en el punto, (m, b) de la tangente y = m x + b)."""
    if ecuacion.count("=") != 1:
        raise ValueError("escribe una ecuación con un solo signo =, por ejemplo: x**2 + y**2 = 25")
    izq, der = ecuacion.split("=")
    F = parsear(izq, ("x", "y")) - parsear(der, ("x", "y"))
    punto = {x: sp.nsimplify(px), y: sp.nsimplify(py)}
    if sp.simplify(F.subs(punto)) != 0:
        raise ValueError(f"el punto ({px}, {py}) no está en la curva: el lado izquierdo menos el derecho vale "
                         f"{cifras(F.subs(punto), 4)}.")
    Fx, Fy = sp.diff(F, x), sp.diff(F, y)      # derivar F = 0 da Fx + Fy·y' = 0
    if sp.simplify(Fy.subs(punto)) == 0:
        raise ValueError("el coeficiente de y' se anula en ese punto: la fórmula no da la pendiente "
                         "(posible tangente vertical).")
    dydx = sp.simplify(-Fx / Fy)
    m = sp.simplify(dydx.subs(punto))
    return dydx, m, (m, sp.simplify(punto[y] - m * punto[x]))


def calculadora_implicita(ecuacion, px, py, n_cifras):
    try:
        dydx, m, (mm, b) = implicita(ecuacion, px, py)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print("y' =", dydx)
    print(f"en ({px}, {py}): y' = {m} ≈ {cifras(m, n_cifras)}")
    print(f"tangente: y = {cifras(mm, n_cifras)} x + {cifras(b, n_cifras)}   ({n_cifras} cifras significativas)")


widgets.interact(calculadora_implicita,
    ecuacion=widgets.Text(value="x**2 + y**2 = 25", description="ecuación"),
    px=widgets.FloatText(value=3.0, description="x0"), py=widgets.FloatText(value=4.0, description="y0"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 6 (resultado conocido)
PRUEBAS_6 = [
    ("circunferencia en (3, 4): -3/4", lambda: implicita("x**2 + y**2 = 25", 3, 4)[1] == sp.Rational(-3, 4)),
    ("circunferencia en (3, -4): 3/4", lambda: implicita("x**2 + y**2 = 25", 3, -4)[1] == sp.Rational(3, 4)),
    ("xy = 12 en (3, 4): -4/3", lambda: implicita("x*y = 12", 3, 4)[1] == sp.Rational(-4, 3)),
    ("folio x³ + y³ = 6xy en (3, 3): -1", lambda: implicita("x**3 + y**3 = 6*x*y", 3, 3)[1] == -1),
    ("x² + xy + y² = 7 en (1, 2): -4/5", lambda: implicita("x**2 + x*y + y**2 = 7", 1, 2)[1] == sp.Rational(-4, 5)),
    ("circunferencia en (5, 0): tangente vertical, se avisa", lambda: _rechaza(lambda: implicita("x**2 + y**2 = 25", 5, 0))),
    ("punto fuera de la curva se rechaza", lambda: _rechaza(lambda: implicita("x**2 + y**2 = 25", 1, 1))),
]
for nombre, prueba in PRUEBAS_6:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 6).** Calcula a mano la pendiente de $x^2+4y^2=8$ en $(2,1)$ y escríbela como número o fracción.

In [ ]:
mi_pendiente = None      # escribe un número o un texto como "-2/3"

ref = implicita("x**2 + 4*y**2 = 8", 2, 1)[1]
if mi_pendiente is None:                 # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    print("La calculadora da:", ref)
    print("coincide" if iguales(sp.nsimplify(mi_pendiente), ref) else "NO coincide: d/dx(4y²) = 8y·y'")

## 7. Derivación logarítmica

Si $y=f(x)>0$: $\ln y=\ln f(x)$, $\dfrac{y'}{y}=\big(\ln f(x)\big)'$ y $y'=f(x)\,\big(\ln f(x)\big)'$. Sirve para potencias con exponente variable, como $x^x$, y para productos y cocientes de muchas potencias.

**Cambio relativo.** Si $y=k\,x_1^{a}x_2^{b}\cdots$, entonces $\dfrac{\Delta y}{y}\approx a\dfrac{\Delta x_1}{x_1}+b\dfrac{\Delta x_2}{x_2}+\cdots$ En el peor caso se suman los valores absolutos. El aporte de cada medición es su exponente por su error relativo.

Dos calculadoras: la primera aplica la derivación logarítmica a una función positiva; la segunda reparte el error relativo de un resultado entre las mediciones.

In [ ]:
def derivada_logaritmica(f_txt):
    """(ln f simplificado, (ln f)', f' = f·(ln f)')."""
    f = parsear(f_txt)
    xp = sp.symbols("xp", positive=True)        # el método supone f > 0; se trabaja con x > 0
    fp = f.subs(x, xp)
    lnf = sp.expand_log(sp.log(fp), force=True)
    dlnf = sp.simplify(sp.diff(lnf, xp))
    return lnf.subs(xp, x), dlnf.subs(xp, x), sp.simplify(fp * dlnf).subs(xp, x)


def sensibilidad(exponentes, errores_rel):
    """Aportes |a|·|e| de cada medición (ordenados de mayor a menor) y total en el peor caso."""
    if len(exponentes) != len(errores_rel) or not exponentes:
        raise ValueError("debe haber un exponente por cada error relativo.")
    aportes = sorted(((abs(a_) * abs(e_), i + 1) for i, (a_, e_) in enumerate(zip(exponentes, errores_rel))), reverse=True)
    return aportes, sum(a_ for a_, _ in aportes)


def calculadora_logaritmica(f_txt, a, n_cifras):
    try:
        lnf, dlnf, dfx = derivada_logaritmica(f_txt)
        va = evaluar_en(dfx, x, a)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print("ln f      =", lnf)
    print("(ln f)'   =", dlnf)
    print("f' = f·(ln f)' =", dfx)
    print(f"f'({a}) = {cifras(va, n_cifras)}  ({n_cifras} cifras significativas)"
          f"   | cociente con h = 1e-6: {cifras(cociente_num(parsear(f_txt), x, a), n_cifras)}")


widgets.interact(calculadora_logaritmica,
    f_txt=widgets.Text(value="x**x", description="f(x) ="),
    a=widgets.FloatText(value=2.0, description="a (> 0)"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));


def calculadora_sensibilidad(a1, e1, a2, e2, a3, e3):
    exps = [v for v in (a1, a2, a3)]
    errs = [v / 100 for v in (e1, e2, e3)]
    try:
        aportes, total = sensibilidad(exps, errs)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    for ap, i in aportes:
        print(f"medición {i}: exponente {exps[i - 1]:g} × error {100 * errs[i - 1]:.4g} % = {100 * ap:.4g} %")
    print(f"error relativo del resultado (peor caso): {100 * total:.4g} %")


print("Calculadora de sensibilidad: exponente y error relativo (%) de cada medición; usa 0 si no hay tercera.")
widgets.interact(calculadora_sensibilidad,
    a1=widgets.FloatText(value=2, description="exponente 1"), e1=widgets.FloatText(value=0.08, description="error 1 (%)"),
    a2=widgets.FloatText(value=1, description="exponente 2"), e2=widgets.FloatText(value=0.4167, description="error 2 (%)"),
    a3=widgets.FloatText(value=0, description="exponente 3"), e3=widgets.FloatText(value=0, description="error 3 (%)"));

In [ ]:
# Casos de prueba de la sección 7 (resultado conocido)
PRUEBAS_7 = [
    ("x^x -> x^x(ln x + 1)", lambda: iguales(derivada_logaritmica("x**x")[2], x**x*(sp.log(x) + 1))),
    ("x^(ln x) en e: 2", lambda: cerca(evaluar_en(derivada_logaritmica("x**log(x)")[2], x, sp.E), 2, 1e-9)),
    ("(x+1)²√x/(x+3)³ en 1: 3/64",
     lambda: evaluar_en(derivada_logaritmica("(x + 1)**2*sqrt(x)/(x + 3)**3")[2], x, 1) == sp.Rational(3, 64)),
    ("eje actual: 2·0.08 % + 0.417 % ≈ 0.577 %", lambda: round(100*sensibilidad([2, 1], [0.0008, 0.5/120])[1], 3) == 0.577),
    ("con micrómetro: ≈ 0.433 %", lambda: round(100*sensibilidad([2, 1], [0.00008, 0.5/120])[1], 3) == 0.433),
    ("P = I²R con 2 % de corriente: 4 %", lambda: cerca(sensibilidad([2, 1], [0.02, 0])[1], 0.04)),
    ("listas de distinto tamaño se rechazan", lambda: _rechaza(lambda: sensibilidad([2, 1], [0.01]))),
]
for nombre, prueba in PRUEBAS_7:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 7).** Con $y=x^{\mathrm{sen}\,x}$, calcula a mano $y'(\pi/2)$ y escribe el número.

In [ ]:
mi_valor = None          # escribe un número

ref = evaluar_en(derivada_logaritmica("x**sin(x)")[2], x, sp.pi/2)
if mi_valor is None:                     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    print("La calculadora da:", ref)
    print("coincide" if cerca(mi_valor, ref, 1e-6) else "NO coincide: ln y = sen x · ln x; deriva con la regla del producto")

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
todas = PRUEBAS_1 + PRUEBAS_2 + PRUEBAS_3 + PRUEBAS_4 + PRUEBAS_5 + PRUEBAS_6 + PRUEBAS_7
fallas = [nombre for nombre, prueba in todas if not prueba()]
assert not fallas, f"Fallan {len(fallas)} pruebas: {fallas}"
print(f"Todas las verificaciones pasaron: {len(todas)}/{len(todas)}")